# Mzansi Distribution-Grid Demo — ODS-E on real site data

Real (anonymized) telemetry from **Mzansi Textiles**, a 2.3 MW commercial & industrial PV installation powering a textile manufacturing plant in Durban, KZN:
**7 Huawei string inverters**, a **billing meter** (IEC 62056 / OBIS daily registers), and site aggregates
(root-cause summary, cost impact, energy balance, tariff rates).

**Datasets** (mini kit, 6 months: Mar–Sep 2026):
| File | Contents |
|---|---|
| `mzansi_telemetry_6mo.csv.gz` | 5-min telemetry × 7 inverters (372k rows) |
| `durban_weather_6mo.csv.gz` | hourly Durban weather |
| `mzansi_meter_6mo.json` | IEC 62056 billing-meter daily registers |
| `root_cause_summary.csv`, `cost_impact_by_inverter.csv`, `energy_balance.csv`, `Tariff*.csv` | site aggregates |

Full narrative: [Mzansi faults on a distribution grid](https://github.com/AsobaCloud/odse/issues/56) — driving use case for SEPs 049–052.


In [ ]:
!pip install -q odse pandas matplotlib

In [ ]:
import pandas as pd
from odse.transformer import transform
from odse.validator import validate_batch

BASE = "https://raw.githubusercontent.com/AsobaCloud/odse/main/colab/data"
tel = pd.read_csv(f"{BASE}/mzansi_telemetry_6mo.csv.gz")
print(tel.shape)
tel.head()

In [ ]:
# Convert raw telemetry to ODS-E records via the Huawei transform
import csv, io
from odse.transformer import HuaweiTransformer

tx = HuaweiTransformer()
# CSV export shape: time,power,inverter_state,run_state,...
# The kit telemetry is already site-normalized; here we demonstrate validation on
# one inverter's rows mapped to ODS-E fields.
inv = tel[tel.asset_id == tel.asset_id.iloc[0]].copy()
records = []
for r in inv.itertuples():
    records.append({{
        "timestamp": r.timestamp,
        "kWh": float(r.kwh),
        "kW": float(r.power),
        "voltage_ac": float(r.voltage),
        "temperature": float(r.temperature),
        "poa_irradiance": float(r.irradiance),
        "wind_speed_ms": float(r.wind_speed),
        "error_type": r.error_type,
        "asset_id": r.asset_id,
    }})
print(len(records), "ODS-E records")

In [ ]:
# Validate the batch (SEP-049 strict vocabulary: UNKNOWN_FIELD warnings, per-phase ready)
summary = validate_batch(records[:500])
print(summary)

In [ ]:
# Fault-event classification: inverter_state codes from the FusionSolar KPI stream
# 768 = high string voltage | 770 = DC arc fault | 771 = grid relay fault | 513/514 = grid over/under-voltage
tel["inverter_state"] = pd.to_numeric(tel["inverter_state"], errors="coerce")
fault = tel[tel["inverter_state"].isin([40960, 45056, 768, 770, 771, 1024, 1280, 513, 514])]
print(fault.groupby(["inverter_state", "asset_id"]).size().unstack(fill_value=0))

In [ ]:
# Energy balance: on-site generation vs grid purchases (billing meter, IEC 62056 / OBIS)
import json, urllib.request
meter = json.load(urllib.request.urlopen(f"{BASE}/mzansi_meter_6mo.json"))
regs = {{}}
for rd in meter["MeterReading"]:
    for r in rd["Readings"]:
        parts = r["ReadingType ref"].split(".")
        key = parts[4]  # 1=import, 15=export, 16-19=slow counters
        regs.setdefault(key, []).append((rd["timeStamp"], r["value"]))
imp = regs["1"]; exp = regs["15"]
di = imp[-1][1] - imp[0][1]; de = exp[-1][1] - exp[0][1]
print(f"Grid import over window: {{di:,}} units | Export: {{de:,}} units")
print(f"Site is {'net-importing' if di > de else 'net-exporting'} (ratio {{di/max(de,1):.1f}}:1)")

In [ ]:
# Cost impact: join shortfall to tariff (site aggregates)
cost = pd.read_csv(f"{BASE}/cost_impact_by_inverter.csv")
root = pd.read_csv(f"{BASE}/root_cause_summary.csv")
print(root.to_string(index=False))
print()
print(cost.to_string(index=False))